# Trimorious M1.4.1 — GPU Teacher Pilot
**GPU-required variant | 100 proposals per teacher | private candidates only**

This variant preserves the original generation, D07 screening, scientific verification and checkpoint logic. It requires explicit license/output-use approval for both pinned teacher models and a CUDA GPU before starting generation. No training or publishing.

In [ ]:
%pip -q install numpy scipy sympy scikit-learn pandas rapidfuzz

## 0. GPU configuration
Upload the M1.3.2 full private archive and M1.4 pilot ZIP to `/content`. Select T4 GPU or better. Review the model licenses and output-use terms at the pinned revisions, then tick both approval boxes below (or change the two approval variables to True). If either is missing, execution stops before generation. Drive is enabled for persistent checkpoints.

- [Qwen3 pinned model](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/tree/cdbee75f17c01a7cc42f958dc650907174af0554)
- [Phi-4-mini pinned model](https://huggingface.co/microsoft/Phi-4-mini-instruct/tree/cfbefacb99257ffa30c83adab238a50856ac3083)

Approval is a user decision, not a legal determination by this notebook.

In [ ]:
# @title GPU teacher pilot settings
import os, json, re, math, random, statistics, hashlib, zipfile, time, csv, sqlite3, gc, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction as F
from datetime import datetime, timezone
import numpy as np
import scipy
from scipy import stats, optimize, integrate
import sympy as sp
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
ROOT=Path(os.getenv('M141_ROOT','/content/trimorious_PRIVATE_m1_4_1'))
D07_ZIP=Path(os.getenv('M141_D07','/content/trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'))
M14_ZIP=Path(os.getenv('M141_PRIOR','/content/Trimorious_M1_4_PRIVATE_Pilot_Validation.zip'))
SEED=417031
RUN_TEACHERS=True
USE_DRIVE=True
TEACHER_LIMIT=100  # per teacher; increase after pilot inspection
TEACHER_MAX_NEW_TOKENS=512
QWEN3_LICENSE_APPROVED = False # @param {type:"boolean"}
PHI4_LICENSE_APPROVED = False # @param {type:"boolean"}
LICENSE_APPROVALS={'qwen3':QWEN3_LICENSE_APPROVED,'phi4':PHI4_LICENSE_APPROVED}
RUN_SEMANTIC=False # optional embedding diversity screen; see section 7
EMBEDDING_REPO=''
EMBEDDING_REVISION=''
EMBEDDING_LICENSE_APPROVED=False
ALLOW_TRAINING=False
ALLOW_PUBLISH=False
assert not ALLOW_TRAINING and not ALLOW_PUBLISH
# Fail closed before spending time generating the CPU corpus.
if not RUN_TEACHERS:
    raise RuntimeError('GPU-only notebook: RUN_TEACHERS must be True.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU or better in Colab, then restart and Run all.')
if not all(LICENSE_APPROVALS.values()):
    raise RuntimeError('Review the pinned teacher licenses and output-use terms, then explicitly approve both models in this configuration cell.')
for required in (D07_ZIP,M14_ZIP):
    if not required.is_file():
        raise FileNotFoundError(f'Upload required archive to /content: {required.name}')
    with zipfile.ZipFile(required) as archive:
        if archive.testzip() is not None:
            raise ValueError(f'Corrupted archive: {required.name}')
print('GPU preflight passed:',torch.cuda.get_device_name(0), '| teacher target:',2*TEACHER_LIMIT)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1')
ROOT.mkdir(parents=True,exist_ok=True)
VERSION='M141-v0.1'
TARGETS={'mathematical_reasoning':1000,'scientific_reasoning':750,'agentic_tool_use':750,'statistical_reasoning':500,'research_methodology':500,'scientific_verification':500,'experimental_design':300,'literature_synthesis':250,'academic_articulation':250,'research_ethics_reliability':200}
DATASETS=dict(zip(TARGETS,['D02','D01','D04','D05','D01','D09','D01','D01','D01','D09']))
def canon(o): return json.dumps(o,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False)
def sha(o): return hashlib.sha256(o if isinstance(o,bytes) else canon(o).encode()).hexdigest()
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def fmt(x): return format(float(x),'.12g')
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
def write_json(p,o): p.write_text(json.dumps(o,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
def write_lines(p,rows): p.write_text(''.join(canon(r)+'\n' for r in rows),encoding='utf-8')
def read_lines(b): return [json.loads(l) for l in b.splitlines() if l.strip()]
print('Target:',sum(TARGETS.values()),'Output root:',ROOT)

## 1. Prior candidate integrity (no inheritance)
Read the M1.4 manifest and prior prompts only. Prior records are not silently added to this new corpus. Preserve original validation candidates and repositories. D07 is opened later by the screening stage, not the generator.

In [ ]:
assert M14_ZIP.is_file(), f'Missing prior archive: {M14_ZIP}'
with zipfile.ZipFile(M14_ZIP) as z:
    assert z.testzip() is None
    mf=json.loads(z.read('archive_manifest.json'))
    assert set(mf)==set(z.namelist())-{'archive_manifest.json'}
    assert all(sha(z.read(n))==h for n,h in mf.items())
    prior_records=[r for n in mf if n.startswith('candidate_datasets/') and n.endswith('.jsonl') for r in read_lines(z.read(n))]
prior_prompts=[r['prompt'] for r in prior_records]
assert len(prior_records)==1000 and len(set(r['id'] for r in prior_records))==1000
prior_exact=set(map(norm,prior_prompts)); prior_masked=set(map(mask,prior_prompts))
print('Prior manifest checked:',len(mf),'files; prior prompts:',len(prior_prompts))

## 2. Computational task mechanisms
The first calculation uses a closed form, exact rational arithmetic or direct algorithm. A second implementation uses an independent identity, symbolic engine or numerical library. Agreement checks only the result; derivation, interpretation and units still require review. Cases are synthetic and bounded.

In [ ]:
def num(q,a,b,why,method,unit='dimensionless',trace=None):
    a,b=float(a),float(b)
    assert math.isfinite(a) and math.isfinite(b)
    return dict(prompt=q,reference_response=f'{why} Final result: {fmt(a)} ({unit}).',numeric_answer=a,independent_reference=b,unit=unit,verification={'passed':close(a,b),'method':method,'status':'numeric_checked_review_pending' if close(a,b) else 'numeric_check_failed'},tool_trace=trace,evidence=[],rubric=['Check derivation and assumptions','Check final value and units'])

def math_case(t,r):
    a,b,c=[r.randint(2,80) for _ in range(3)]
    x=sp.Symbol('x')
    if t==0:
        return num(f'For x>0, solve x + {a}/x = {a+b+1} for the larger root.',((a+b+1)+math.sqrt((a+b+1)**2-4*a))/2,max(map(float,sp.solve(x*x-(a+b+1)*x+a,x))),f'Multiply by x; choose the larger quadratic root since x>0.','quadratic formula / symbolic roots')
    if t==1:
        return num(f"Find the value at t=1 of y solving y'' = {a}, y(0)={b}, y'(0)={c}.",b+c+a/2,integrate.quad(lambda u:c+a*u,0,1)[0]+b,f'Integrate acceleration twice: y(t)={b}+{c}t+({a}/2)t².','analytical / quadrature')
    if t==2:
        return num(f'For integers n={a+b}, k={a}, approximately how many k-element subsets include two specified distinct elements (give a numerical approximation)?',math.comb(a+b-2,a-2),sp.binomial(a+b-2,a-2),'Reserve the specified two elements and choose the remaining k-2 from n-2.','integer combination / SymPy')
    if t==3:
        return num(f'Compute the spectral radius of the triangular matrix [[{a}, {b}], [0, -{c}]].',max(a,c),max(abs(np.linalg.eigvals([[a,b],[0,-c]]))),'A triangular matrix has its diagonal entries as eigenvalues; take the largest absolute value.','diagonal eigenvalues / NumPy')
    if t==4:
        return num(f'On [0,1], find the minimum value of {a}x² - {b}x + {c}.',a*(u:=min(1,b/(2*a)))**2-b*u+c,min(float(optimize.minimize_scalar(lambda x:a*x*x-b*x+c,bounds=(0,1),method='bounded').fun),c,a-b+c) if b<2*a else a-b+c,f'Clamp the stationary point {b}/(2×{a}) to [0,1], then evaluate the objective.','convex stationary point / bounded optimization')
    if t==5:
        return num(f'Calculate the Euclidean distance from point ({a},{b}) to the line 3x+4y={c}.',abs(3*a+4*b-c)/5,np.linalg.norm(np.array([a,b])-(np.array([a,b])-((3*a+4*b-c)/25)*np.array([3,4]))),'Project onto the unit normal (3,4)/5.','point-line formula / projection')
    if t==6:
        return num(f'Compute the remainder of {a}^{b} modulo {c}.',pow(a,b,c),int(sp.Mod(sp.Integer(a)**b,c)),'Use modular exponentiation, reducing intermediate products.','modular exponentiation / exact integer power')
    if t==7:
        return num(f'For a polynomial p(x)=({a}x+{b})²({c}x+1), find the coefficient of x².',a*a+2*a*b*c,sp.expand((a*x+b)**2*(c*x+1)).coeff(x,2),'Collect the x² contributions from the constant and linear terms of the final factor.','coefficient convolution / expansion')
    if t==8:
        return num(f'Find the number of walks of length two from vertex 1 to vertex 3 in a weighted adjacency matrix [[0,{a},{b}],[{a},0,{c}],[{b},{c},0]], interpreting integer weights as parallel edge counts.',a*c,np.linalg.matrix_power(np.array([[0,a,b],[a,0,c],[b,c,0]],dtype=np.int64),2)[0,2],'Only walks via vertex 2 contribute; multiply the two edge counts.','walk enumeration / matrix square')
    return num(f'For a geometric progression starting at {a} with ratio 1/{b}, compute the sum of its first {c} terms.',a*(1-(1/b)**c)/(1-1/b),sum(F(a,b**i) for i in range(c)),'Use the finite geometric series identity, not the infinite limit.','closed form / exact summation')

def science_case(t,r):
    a,b,c=[r.randint(2,90) for _ in range(3)]
    if t==0: return num(f'A cart of mass {a} kg initially travels at {b} m/s. A constant force {c} N opposes motion until rest. Find stopping distance, neglect other forces.',a*b*b/(2*c),b*(a*b/c)/2,'Work by the opposing force equals initial kinetic energy.','work energy / constant acceleration','m')
    if t==1: return num(f'A rigid tank contains ideal gas at {a+280} K and pressure {b} kPa. It is heated to {a+c+280} K at fixed volume and mole count. Find final pressure.',b*(a+c+280)/(a+280),b+b*c/(a+280),'At fixed n,V, pressure is proportional to absolute temperature.','ideal gas ratio / pressure increment','kPa')
    if t==2: return num(f'A tracer decays with half-life {a} hours. Initial activity is {b*100} Bq. Estimate activity after {c} hours.',b*100*2**(-c/a),b*100*math.exp(-math.log(2)*c/a),'Use exponential decay with lambda=ln(2)/half-life.','half-life / exponential','Bq')
    if t==3: return num(f'A square membrane {a} cm per side supports uniform pressure difference {b} Pa. Find total normal force.',b*(a/100)**2,b*a*a/10000,'Force is pressure times area; convert centimeters to meters.','SI area / cm² conversion','N')
    if t==4: return num(f'A well-mixed vessel contains {a+b} L solution at {c} mg/L. Remove {a} L, replace with solute-free water, mix. What is final concentration?',c*b/(a+b),(c*(a+b)-c*a)/(a+b),'Conserve solute mass through withdrawal and dilution.','remaining mass / removed mass','mg/L')
    if t==5: return num(f'An elastic spring of stiffness {a*10} N/m is compressed from {b/100:g} m to {(b+c)/100:g} m. Find the increase in stored energy.',5*a*((b+c)**2-b*b)/10000,integrate.quad(lambda x:a*10*x,b/100,(b+c)/100)[0],'Integrate Hooke restoring-force magnitude over the compression interval.','energy difference / quadrature','J')
    if t==6: return num(f'A hypothetical enzyme follows Michaelis–Menten kinetics with Vmax {a} µmol/min, Km {b} mM, substrate {c} mM. Compute rate.',a*c/(b+c),a/(1+b/c),'Apply the stated saturating kinetic model.','rate law / reciprocal form','µmol/min')
    if t==7: return num(f'A uniform film of thickness {a} µm has thermal conductivity {b/10:g} W/(m K), area {c/100:g} m². Find thermal resistance for one-dimensional conduction.',a*1e-6/((b/10)*(c/100)),a/(1000*b*c),'Use thickness divided by conductivity times area.','SI formula / simplified units','K/W')
    if t==8: return num(f'A hypothetical population grows at continuous rate {a/1000:g} per day from {b*100} cells. Predict expected count at day {c}.',b*100*math.exp(a*c/1000),b*100*float(sp.exp(sp.Rational(a*c,1000))),'Apply the specified exponential model; this is a prediction, not a measured count.','exponential / symbolic exponential','cells')
    return num(f'An ideal hydraulic piston of input area {a} cm² receives {b} N; output area {c} cm². Find output force with no losses.',b*c/a,float(F(b*c,a)),'Equal pressure implies force proportional to piston area.','pressure transfer / rational arithmetic','N')

def stats_case(t,r):
    a,b,c=[r.randint(2,90) for _ in range(3)]
    n=a+b+c; v=[a,b,c,a+b,b+c]
    if t==0: return num(f'A sample of size {n} has {a} events. Calculate the Wilson 95% interval center using z=1.96 (not its midpoint rounded to percent).',(a/n+1.96**2/(2*n))/(1+1.96**2/n),(a+1.96**2/2)/(n+1.96**2),'Apply the Wilson center formula to the event fraction.','Wilson formula / count form')
    if t==1: return num(f'An independent sample contains {a} observations with sample SD {b}. Another has {c} observations with sample SD {b+1}. Compute standard error of the difference of means.',math.sqrt(b*b/a+(b+1)**2/c),np.linalg.norm([b/math.sqrt(a),(b+1)/math.sqrt(c)]),'Sum independent estimated variances of the means.','variance addition / vector norm')
    if t==2: return num(f'In a finite lot of {n} items, {a} are defective. Draw two without replacement. Find probability exactly one is defective.',a*(n-a)/math.comb(n,2),stats.hypergeom.pmf(1,n,a,2),'Count unordered pairs containing one defective and one nondefective.','combinatorics / hypergeometric PMF')
    if t==3: return num(f'Given outcomes {v}, compute population variance (divide by N).',sum((i-sum(v)/len(v))**2 for i in v)/len(v),np.var(v),'Center at the arithmetic mean, then average squared deviations.','direct variance / NumPy')
    if t==4: return num(f'Assume Poisson rate {a/10:g} events per hour. What is expected waiting time until the next event?',10/a,stats.expon.mean(scale=10/a),'Under a homogeneous Poisson process, waiting time has exponential mean 1/rate.','reciprocal rate / SciPy','h')
    if t==5: return num(f'For independent binomial counts with n={a+b}, p={c/100:g}, compute count variance.',(a+b)*(c/100)*(1-c/100),stats.binom.var(a+b,c/100),'Add Bernoulli variances.','binomial formula / SciPy')
    if t==6: return num(f'An independent two-group study has risks {a/100:g} and {b/100:g}. Calculate absolute risk difference A minus B.',(a-b)/100,float(F(a,100)-F(b,100)),'Subtract the second risk; keep the sign.','risk difference / rational arithmetic')
    if t==7: return num(f'Predicted values are {v}; observed values are {[i+(-1)**j*c for j,i in enumerate(v)]}. Calculate mean absolute error.',c,np.mean(np.abs(np.array(v)-np.array([i+(-1)**j*c for j,i in enumerate(v)]))),'Take absolute residuals before averaging.','residual pattern / NumPy')
    if t==8: return num(f'A categorical distribution has unnormalized masses [{a},{b},{c}]. Find its Shannon entropy in natural logarithm units.',-sum(i/n*math.log(i/n) for i in [a,b,c]),stats.entropy([a,b,c]),'Normalize masses, then sum -p ln p.','direct entropy / SciPy','nats')
    return num(f'Two independent estimates are {a} and {b} with known variances {c} and {c+1}. Find inverse-variance weighted mean.',(a/c+b/(c+1))/(1/c+1/(c+1)),np.average([a,b],weights=[1/c,1/(c+1)]),'Weight each estimate by precision (inverse variance).','precision formula / NumPy')

## 3. D04 / D10 — executed multi-tool workflows
Each synthetic workflow has at least two successful calls. The second call consumes the first observation. Some workflows intentionally exercise a rejected argument and bounded recovery. Only fixed registered operations run on generator-owned inputs. No arbitrary code execution, no model-supplied code, and no external services. The separate verifier replays each step with alternative formulas and verifies the final result.

In [ ]:
def tool(name,a):
    if name=='sum': return float(np.sum(a['values']))
    if name=='mean': return float(np.mean(a['values']))
    if name=='variance': return float(np.var(a['values'],ddof=0))
    if name=='sqrt':
        if a['x']<0: raise ValueError('negative radicand')
        return float(np.sqrt(a['x']))
    if name=='ratio':
        if a['den']==0: raise ValueError('zero denominator')
        return float(np.divide(a['num'],a['den']))
    if name=='difference': return float(np.subtract(a['a'],a['b']))
    if name=='product': return float(np.prod(a['values']))
    if name=='log':
        if a['x']<=0: raise ValueError('nonpositive logarithm')
        return float(np.log(a['x']))
    if name=='trapezoid': return float(integrate.trapezoid(a['y'],a['x']))
    if name=='slope': return float(stats.linregress(a['x'],a['y']).slope)
    if name=='tail': return float(stats.binom.sf(a['k']-1,a['n'],a['p']))
    raise ValueError('unknown tool')
def tool_ref(name,a):
    if name=='sum': return sum(a['values'])
    if name=='mean': return statistics.mean(a['values'])
    if name=='variance': return statistics.pvariance(a['values'])
    if name=='sqrt': return math.sqrt(a['x']) if a['x']>=0 else (_ for _ in ()).throw(ValueError('negative radicand'))
    if name=='ratio':
        if a['den']==0: raise ValueError('zero denominator')
        return a['num']/a['den']
    if name=='difference': return a['a']-a['b']
    if name=='product': return math.prod(a['values'])
    if name=='log': return math.log(a['x'])
    if name=='trapezoid': return sum((x2-x1)*(y1+y2)/2 for x1,x2,y1,y2 in zip(a['x'],a['x'][1:],a['y'],a['y'][1:]))
    if name=='slope':
        x,y=a['x'],a['y']; mx,my=statistics.mean(x),statistics.mean(y)
        return sum((i-mx)*(j-my) for i,j in zip(x,y))/sum((i-mx)**2 for i in x)
    if name=='tail': return sum(math.comb(a['n'],j)*a['p']**j*(1-a['p'])**(a['n']-j) for j in range(a['k'],a['n']+1))
    raise ValueError(name)
TOOL_NAMES=['sum','mean','variance','sqrt','ratio','difference','product','log','trapezoid','slope','tail']
def agent_case(t,r):
    a,b,c=[r.randint(2,90) for _ in range(3)]; v=[a,b,c,a+c]; trace=[]
    def call(name,args,depends=None,expected_error=False):
        try:
            obs=tool(name,args)
        except ValueError as e:
            if not expected_error: raise
            try: tool_ref(name,args)
            except ValueError: pass
            else: raise AssertionError('Reference failed to reject invalid arguments')
            trace.append({'step':len(trace)+1,'tool':name,'arguments':args,'depends_on':depends,'status':'rejected','error':str(e),'replay_pass':True})
            return None
        assert not expected_error
        ref=tool_ref(name,args); replay=tool(name,args)
        assert close(obs,ref) and close(obs,replay)
        trace.append({'step':len(trace)+1,'tool':name,'arguments':args,'depends_on':depends,'status':'executed','observation':obs,'independent_reference':float(ref),'replay_pass':True})
        return obs
    if t==0:
        q=f'For readings {v}, calculate population standard deviation via variance then square root.'; x=call('variance',{'values':v}); y=call('sqrt',{'x':x},1); ref=statistics.pstdev(v)
    elif t==1:
        q=f'Integrate flow samples {v} at times [0,1,3,5], then divide integrated volume by elapsed time to obtain time-average flow.'; x=call('trapezoid',{'x':[0,1,3,5],'y':v}); y=call('ratio',{'num':x,'den':5},1); ref=(v[0]+3*v[1]+4*v[2]+2*v[3])/10
    elif t==2:
        q=f'Calculate total mass from components {v}, then find the fraction due to the first component.'; x=call('sum',{'values':v}); y=call('ratio',{'num':a,'den':x},1); ref=a/sum(v)
    elif t==3:
        q=f'Fit a line to x=[0,1,2,3], y={v}, then predict the change for an additional {c} x units using its slope.'; x=call('slope',{'x':[0,1,2,3],'y':v}); y=call('product',{'values':[x,c]},1); ref=c*sum((i-1.5)*j for i,j in enumerate(v))/5
    elif t==4:
        q=f'Compute the mean of readings {v}, then calculate its difference from target {b}.'; x=call('mean',{'values':v}); y=call('difference',{'a':x,'b':b},1); ref=sum(v)/4-b
    elif t==5:
        q=f'With positive activities {a} and {b}, calculate the natural log of their ratio.'; x=call('ratio',{'num':a,'den':b}); y=call('log',{'x':x},1); ref=math.log(a)-math.log(b)
    elif t==6:
        q=f'An instrument exported counts {v} and an invalid zero duration. Demonstrate rejection of division by zero; use independently supplied corrected duration {c} to compute aggregate count rate.'; x=call('sum',{'values':v}); call('ratio',{'num':x,'den':0},1,True); y=call('ratio',{'num':x,'den':c},1); ref=sum(v)/c
    elif t==7:
        q=f'Calculate the geometric mean of positive lengths {a} and {b}. First demonstrate rejection of a negative product supplied in error, then use the correct positive product.'; x=call('product',{'values':[a,b]}); call('sqrt',{'x':-x},1,True); y=call('sqrt',{'x':x},1); ref=math.sqrt(a)*math.sqrt(b)
    elif t==8:
        q=f'For independent Bernoulli trials n={a}, success probability {b/100:g}, compute P(at least 2 successes), then its complement.'; x=call('tail',{'n':a,'k':2,'p':b/100}); y=call('difference',{'a':1,'b':x},1); ref=(1-b/100)**a+a*(b/100)*(1-b/100)**(a-1)
    else:
        q=f'For two positive readings {a} and {b}, compute their ratio and multiply it by calibration factor {c}.'; x=call('ratio',{'num':a,'den':b}); y=call('product',{'values':[x,c]},1); ref=a*c/b
    assert close(y,ref)
    explanation='Executed '+ ' → '.join(s['tool']+(' (rejected invalid input)' if s['status']=='rejected' else f"={fmt(s['observation'])}") for s in trace)+'.'
    p=num(q,y,ref,explanation,'registered workflow / independent final formula',trace=trace)
    p['verification']['status']='numeric_and_multitool_checked_review_pending'
    return p

## 4. Context-specific scientific narrative cases
These are authored synthetic evidence scenarios with case-specific data and rubrics. No claim of real research or published sources is made. Reviewers must verify the reasoning and evidence linkage; varying numbers does not by itself constitute intellectual diversity. Teacher refinement remains separate.

In [ ]:
DOMAINS=[('battery cycling','capacity retention','cells'),('river monitoring','turbidity','sites'),('plant germination','germination rate','trays'),('ceramic testing','fracture strength','specimens'),('acoustic testing','sound attenuation','panels'),('solar coating','reflectance','coupons'),('robot positioning','position error','trials'),('water filtration','solute removal','filters'),('thermal insulation','heat flux','assemblies'),('sensor calibration','measurement error','devices')]
def narrative_case(f,t,r,i):
    domain,outcome,unit=DOMAINS[i%len(DOMAINS)]
    n=r.randint(24,150); a=r.randint(2,20); b=r.randint(21,60); c=r.randint(1,10)
    if f=='research_methodology':
        cases=[
          (f'{n} {unit} were assigned by site convenience; site A received treatment and site B control. The outcome is {outcome}.',f'Assignment of all {n} {unit} is confounded with site. Randomize within site or use multiple sites per arm; prespecify {outcome} and analyze clustering. The current comparison is not a treatment effect.'),
          (f'{n} {unit} each provided {a} repeated readings; the analysis treated every reading as independent.',f'There are {n} independent units at most, not {n*a} independent observations. Use a hierarchical or repeated-measures model for {outcome}; estimate uncertainty at the unit level.'),
          (f'Investigators changed the primary outcome to {outcome} after inspecting {a} candidate endpoints from {n} {unit}.',f'The selection among {a} endpoints after inspecting {n} units makes nominal confirmatory inference unreliable. Label the result exploratory, report all endpoints and confirm {outcome} in an independent prespecified study.'),
          (f'{a} of {n} {unit} lacked outcome values after treatment; missingness was not investigated.',f'The missing outcome fraction is {a}/{n}. Compare missingness by arm and baseline variables, justify a missing-data model, and report sensitivity analyses; the complete-case estimate of {outcome} may be biased.'),
          (f'The team tuned a prediction model on all {n} {unit} before evaluating it on a subset of {b} of them.',f'The {b}-unit subset is not independent of tuning on {n} units. Split by independent {unit} before preprocessing and tuning; evaluate {outcome} on an untouched test partition.')]
    elif f=='scientific_verification':
        cases=[
          (f'A simulation of {domain} uses {n} time steps; doubling resolution changes the output by {a}%.',f'A {a}% change on refinement means convergence at {n} steps is not established. Run further refinements, report an error tolerance and compare with an analytical limit or independent solver.'),
          (f'Two instruments measure {outcome}; one has a known offset of {a} units, and the observed difference is {b} units.',f'Correct the known {a}-unit offset in the stated direction before interpreting the {b}-unit difference; obtain calibration uncertainty and paired measurements. Do not infer agreement from the difference alone.'),
          (f'A script for {domain} reports {n} runs but records seeds for only {a}.',f'Only {a} of {n} runs have recorded seeds. Recover seeds, dependency versions, input hashes and configuration or explicitly mark missing runs unreproducible.'),
          (f'A code output lists {b} successful runs of {n}, with the other runs omitted from the report.',f'Reconcile all {n} runs: {n-b if n>=b else "the remaining"} unsuccessful or unreported runs must be documented if the count is valid. Inspect logs and prespecify exclusions before estimating performance.'),
          (f'A theoretical model of {outcome} assumes a closed system, but a sensitivity run introduces a loss rate of {a}% per interval over {c} intervals.',f'The loss scenario violates the closed-system assumption. Check conservation residuals over all {c} intervals and report sensitivity to the {a}% loss; do not treat the original model as validated for that regime.')]
    elif f=='experimental_design':
        cases=[
          (f'Plan a two-arm experiment with {n} {unit} across {c} batches to estimate a treatment effect on {outcome}.',f'Randomize within each of the {c} batches, balance arms where possible among {n} units, blind outcome assessment, prespecify the effect and analysis; account for batch and justify sample size using variance and target effect.'),
          (f'The proposed study has {n} {unit} and {a} repeated measurements per unit. Design a study for change in {outcome}.',f'Collect a baseline and prespecified follow-up readings on each of {n} units. Use within-unit changes or a mixed model; {a} repeated measures improve within-unit precision but do not create independent units.'),
          (f'A calibration project has budget for {n} measurements and {c} instruments. Design assessment of {outcome} across instruments.',f'Allocate the {n} measurements across {c} instruments, randomized reference levels and replicate runs. Compare bias and repeatability with uncertainty, using a traceable reference.'),
          (f'A planned factorial study of {domain} has two factors with two levels each and {n} available {unit}.',f'Use four factorial cells, approximately {n//4} units each plus balanced remainder. Randomize and estimate main effects and interaction on {outcome}; prespecify analysis and check assumptions.'),
          (f'A pilot of {domain} has {n} available {unit} and suspected drift across {c} time blocks.',f'Block allocation across {c} time periods, randomize within each block, insert reference checks, and model block effects when estimating {outcome}. Avoid assigning one arm exclusively to one time period.')]
    elif f=='literature_synthesis':
        cases=[
          (f'[A] A simulated model of {domain} used {n} parameter settings. [B] A fictional laboratory pilot used {a} independent {unit} but reported no uncertainty.',f'[A] describes computational behavior over {n} settings; [B] provides limited laboratory evidence from {a} units. The evidence types are complementary but not sufficient to establish generalizable performance of {outcome}. A larger independently assessed experiment with uncertainty is needed.'),
          (f'[A] A fictional controlled test of {outcome} used {n} {unit}. [B] A fictional observational survey used {b} units in a different setting.',f'[A] may support a controlled comparison within its conditions; [B] describes observations in a different setting with {b} units. Do not pool effects without comparable definitions and estimates; assess transferability of the {n}-unit test.'),
          (f'[A] A fictional test measured {outcome} at day {a}. [B] A second fictional test measured at day {b}; neither reports intermediate measurements.',f'[A] and [B] refer to different follow-up times ({a} and {b} days). Their results cannot establish a time trajectory or direct effect difference without common measurements and uncertainty.'),
          (f'[A] A fictional protocol proposes {n} {unit} but no results. [B] A fictional pilot documents {a} completed units.',f'[A] provides planned methods rather than findings; [B] documents {a} completed units. The proposed {n}-unit study cannot be treated as evidence of efficacy or validation.'),
          (f'[A] A fictional study reports {n} {unit} from one site. [B] A fictional replication plan proposes {c} sites without completed observations.',f'[A] establishes evidence only within the single-site setting as described; [B] is a plan for {c} sites. External validity remains untested, and no replication outcome should be attributed to [B].')]
    elif f=='academic_articulation':
        cases=[
          (f'A fictional simulation used {n} parameter settings and did not collect physical {outcome} observations.',f'The simulation explored {n} parameter settings for {domain}. The outputs characterize the specified model; physical {outcome} was not measured.'),
          (f'A fictional planned experiment specifies {n} {unit}; {a} units are reserved for calibration, no runs completed.',f'The proposed experiment allocates {a} of {n} {unit} to calibration. No experimental results are available because execution has not begun.'),
          (f'A fictional test completed {n} runs; {a} had incomplete metadata. No missingness analysis was performed.',f'The test completed {n} runs, of which {a} had incomplete metadata. The impact of missing metadata on interpretation of {outcome} has not been evaluated.'),
          (f'A fictional evaluation recorded median runtime {b} ms across {n} runs on a single device.',f'The median recorded runtime was {b} ms across {n} runs on the evaluated device. Performance on other hardware remains untested.'),
          (f'A fictional observational study assessed {outcome} in {n} {unit} at {c} sites; allocation was not randomized.',f'The observational study assessed {outcome} in {n} {unit} across {c} sites. Because allocation was not randomized, causal interpretation requires additional assumptions and evidence.')]
    else:
        cases=[
          (f'The dataset for {domain} includes {n} records, {a} of which have unresolved redistribution rights.',f'Quarantine the {a} unresolved records and verify rights before any redistribution or training. Retain provenance for all {n} records; missing permission is not consent.'),
          (f'The draft reports {b} successful tests but {a} unfavorable outcomes were deliberately excluded without a protocol.',f'Disclose the {a} unfavorable outcomes alongside the {b} successes, preserve the raw records, document exclusions and reassess conclusions without selective reporting.'),
          (f'A colleague proposes presenting {n} simulated outputs as observed {outcome} measurements.',f'Label all {n} outputs as simulated, document model assumptions and validation status, and do not present them as physical observations.'),
          (f'A manuscript cites {a} sources generated by an AI assistant; none were independently checked.',f'Verify each of the {a} citations against the original source and confirm it supports the associated claim. Remove unverifiable references; do not invent replacements.'),
          (f'A replication report for {domain} has {n} planned tests and {a} completed tests, but claims that all passed.',f'Report only the {a} completed tests with their actual outcomes and distinguish the remaining {n-a} planned tests. Do not claim universal success without evidence.')]
    # ensure stated counts are internally possible for the verification omitted-run case
    case,answer=cases[t]
    if (f=='scientific_verification' and t==3 and b>n) or (f=='research_methodology' and t==4 and b>=n): return None
    prompt=f'Fictional {domain} research case ({outcome}): {case} '
    prompt+=('Synthesize only the supplied evidence cards with [A] and [B].' if f=='literature_synthesis' else 'Write a cautious academic statement using only the supplied record.' if f=='academic_articulation' else 'Give a defensible response, explicitly referencing the case facts and limitations.')
    return dict(prompt=prompt,reference_response=answer,numeric_answer=None,independent_reference=None,unit=None,verification={'passed':None,'method':'human evidence/rubric assessment','status':'rubric_review_pending'},tool_trace=None,evidence=[{'id':'synthetic_case','kind':'fictional_case','text':case}],rubric=['Preserve case quantities and evidence status','Address the specific methodological or scientific issue','Do not fabricate observations, sources or certainty'])

## 5. Generate exact target counts and record every disposition
Generate new prompts with exact and number-masked prior-pilot checks. Number masking is a coarse template check, not proof of semantic uniqueness. All generated records receive stable content IDs, provenance, separate verification status and explicit nonapproval. A failed check is quarantined, not silently removed.

In [ ]:
GENERATORS={'mathematical_reasoning':math_case,'scientific_reasoning':science_case,'statistical_reasoning':stats_case,'agentic_tool_use':agent_case}
rng=random.Random(SEED); records=[]; seen=set(); rejected_attempts=Counter()
for family,target in TARGETS.items():
    nt=10 if family in GENERATORS else 5
    for i in range(target):
        t=i%nt
        for attempt in range(2000):
            p=GENERATORS[family](t,rng) if family in GENERATORS else narrative_case(family,t,rng,i//nt+attempt)
            if p is None: rejected_attempts['invalid_scenario']+=1; continue
            q=norm(p['prompt'])
            if q in seen or q in prior_exact or mask(q) in prior_masked:
                rejected_attempts['duplicate_or_prior_template']+=1; continue
            seen.add(q); break
        else: raise RuntimeError(f'Cannot generate unique prompt for {family}/{t}')
        rec={**p,'id':f'M141-{len(records)+1:05d}','family':family,'dataset':DATASETS[family],'template_id':f'{family}_{t:02d}','provenance':{'kind':'original synthetic case','generator':VERSION,'seed':SEED,'source_license':'redistribution review pending','external_source':None},'training_approved':False,'human_decision':'pending'}
        rec['record_sha256']=sha(rec); records.append(rec)
assert len(records)==5000 and Counter(r['family'] for r in records)==Counter(TARGETS)
CORPUS_HASH=sha([r['record_sha256'] for r in records]); print('Corpus:',CORPUS_HASH,'templates:',len({r['template_id'] for r in records}),'rejected attempts:',dict(rejected_attempts))
print(Counter(r['verification']['status'] for r in records))

## 6. Mandatory frozen D07 gate and prior-corpus similarity
Validate the exact frozen question bytes and benchmark manifest. Do not read gold keys or send any evaluation material to teachers. Screen prompts with normalized exact, number-masked, character and word TF-IDF cosine similarity. Flagged examples are quarantined; the 5,000 target is a generated count, not an approval or release guarantee. Prior-pilot similarity is reported separately for review; future merging must resolve cross-version leakage.

In [ ]:
EXPECTED_BENCHMARK='47694f1b2f4dbd44e8f5b6e58962ad07af5c2a844e375249e74d785de0b51283'
EXPECTED_QUESTIONS='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
def load_d07_questions():
    assert D07_ZIP.is_file(),f'Missing D07 archive: {D07_ZIP}'
    with zipfile.ZipFile(D07_ZIP) as z:
        mf=json.loads(z.read('PRIVATE_DO_NOT_TRAIN/manifest.json'))
        raw=z.read('PRIVATE_DO_NOT_TRAIN/D07_questions.jsonl')
        assert sha(raw)==EXPECTED_QUESTIONS and mf['benchmark_sha256']==EXPECTED_BENCHMARK and mf['training_use'] is False
        assert mf['file_hashes']['D07_questions.jsonl']==EXPECTED_QUESTIONS
        qs=[r['question'] for r in read_lines(raw)]
    assert len(qs)==1000 and len(set(qs))==1000
    return qs

def similarities(queries,refs,analyzer,ngram):
    v=TfidfVectorizer(analyzer=analyzer,ngram_range=ngram,sublinear_tf=True,dtype=np.float32)
    z=v.fit_transform([*refs,*queries]); a=z[:len(refs)]; b=z[len(refs):]
    result=[]
    for i in range(0,b.shape[0],256): result.extend(np.asarray((b[i:i+256]@a.T).max(axis=1).toarray()).ravel().tolist())
    return result

def d07_screen(texts):
    refs=load_d07_questions(); nr=set(map(norm,refs)); mr=set(map(mask,refs))
    queries=list(map(mask,texts)); masked=list(map(mask,refs))
    cs=similarities(queries,masked,'char',(3,5)); ws=similarities(queries,masked,'word',(1,2))
    return [{'exact':norm(q) in nr,'masked':mask(q) in mr,'char':float(c),'word':float(w),'flagged':bool(norm(q) in nr or mask(q) in mr or c>=.78 or w>=.72)} for q,c,w in zip(texts,cs,ws)]
leak=d07_screen([r['prompt'] for r in records]); prev=similarities([mask(r['prompt']) for r in records],list(map(mask,prior_prompts)),'char',(3,5))
for r,l,p in zip(records,leak,prev):
    r['d07_screen']=l; r['prior_pilot_char_similarity']=float(p)
    r['disposition']='quarantine' if l['flagged'] or r['verification']['passed'] is False else 'candidate_review_pending'
print('D07 flagged:',sum(x['flagged'] for x in leak),'max char:',max(x['char'] for x in leak),'max word:',max(x['word'] for x in leak))

## 7. Diversity audit and group-disjoint splits
Use exact response counts, number-masked response counts, and nearest-neighbor lexical similarities for prompt and response text. Optional sentence embeddings provide a semantic triage signal, not a scientific correctness classifier. Default CPU run does not download an embedding model. When enabled, provide a pinned reviewed embedding model, with its own license approval. Keep all examples of each mechanism/template in one split; optional teacher proposals target development only. The split is provisional until semantic/source review; no final benchmark is manufactured here.

In [ ]:
def nearest_scores(texts):
    v=TfidfVectorizer(analyzer='char',ngram_range=(3,5),dtype=np.float32).fit_transform(texts)
    nn=NearestNeighbors(n_neighbors=2,metric='cosine',algorithm='brute').fit(v)
    dist,idx=nn.kneighbors(v)
    return [float(max(0,1-next(d for d,j in zip(ds,js) if j!=i))) for i,(ds,js) in enumerate(zip(dist,idx))]
for f in TARGETS:
    group=[r for r in records if r['family']==f]
    ps=nearest_scores([mask(r['prompt']) for r in group]); ans=nearest_scores([mask(r['reference_response']) for r in group])
    for r,p,a in zip(group,ps,ans): r['diversity']={'nearest_masked_prompt':p,'nearest_masked_response':a,'semantic_review':'pending'}
semantic_status='not_run'
if RUN_SEMANTIC:
    assert EMBEDDING_LICENSE_APPROVED and EMBEDDING_REPO and re.fullmatch(r'[0-9a-f]{40}',EMBEDDING_REVISION)
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','-q','install','sentence-transformers'])
    from sentence_transformers import SentenceTransformer
    enc=SentenceTransformer(EMBEDDING_REPO,revision=EMBEDDING_REVISION,trust_remote_code=False)
    for f in TARGETS:
        group=[r for r in records if r['family']==f]
        for field in ['prompt','reference_response']:
            em=enc.encode([r[field] for r in group],normalize_embeddings=True,batch_size=32,show_progress_bar=True)
            d,idx=NearestNeighbors(n_neighbors=2,metric='cosine').fit(em).kneighbors(em)
            for i,r in enumerate(group): r['diversity']['embedding_'+field]=float(1-next(x for x,j in zip(d[i],idx[i]) if j!=i))
    del enc; semantic_status='embedding_triage_completed_review_pending'
val_templates=set()
for family in TARGETS:
    groups=sorted({r['template_id'] for r in records if r['family']==family and r['disposition']!='quarantine'})
    if len(groups)>1: val_templates.update(random.Random(SEED+sum(map(ord,family))).sample(groups,max(1,round(.2*len(groups)))))
for r in records:
    r['split']='quarantine' if r['disposition']=='quarantine' else 'validation_candidate' if r['template_id'] in val_templates else 'development_candidate'
dev=[r for r in records if r['split']=='development_candidate']; val=[r for r in records if r['split']=='validation_candidate']; quarantine=[r for r in records if r['split']=='quarantine']
assert set(r['template_id'] for r in dev).isdisjoint(r['template_id'] for r in val)
assert len(dev)+len(val)+len(quarantine)==5000
print('Split:',Counter(r['split'] for r in records))
diversity=[]
for f in TARGETS:
    g=[r for r in records if r['family']==f]
    diversity.append({'family':f,'count':len(g),'templates':len({r['template_id'] for r in g}),'unique_answers':len({norm(r['reference_response']) for r in g}),'unique_masked_answers':len({mask(r['reference_response']) for r in g}),'near_masked_prompt_095':sum(r['diversity']['nearest_masked_prompt']>=.95 for r in g),'near_masked_answer_095':sum(r['diversity']['nearest_masked_response']>=.95 for r in g)})
print(pd.DataFrame(diversity).to_string(index=False))

## 8. Teacher-model orchestration (optional GPU; not auto-approval)
Two previously evaluated model revisions are configured. They are candidate teachers, not proven superior to the student in all domains. Qwen handles math/statistics/agentic/science; Phi handles research/methodology/writing/verification. A balanced subset of development candidates is routed to each. Both models may be used as challengers via adding families to the route. Teacher sees the original prompt and supplied fictional evidence only, not reference answers or D07. All model text is data; no tool calls from model text are executed. A numeric final value is checked independently; narrative text requires human review even if a teacher agrees. Teacher output is screened against D07 prompts, stored separately, and is never automatically accepted. Review licenses and model-output-use terms before enabling. GPU inference is resumable with a private SQLite database, keyed by corpus, exact revision, prompt version, token limit and runtime versions. Interrupted runs retry incomplete items.

In [ ]:
TEACHERS={
 'qwen3':{'repo':'Qwen/Qwen3-4B-Instruct-2507','revision':'cdbee75f17c01a7cc42f958dc650907174af0554','license_file':'LICENSE','license_sha256':'832dd9e00a68dd83b3c3fb9f5588dad7dcf337a0db50f7d9483f310cd292e92e','families':['mathematical_reasoning','scientific_reasoning','statistical_reasoning','agentic_tool_use','scientific_verification']},
 'phi4':{'repo':'microsoft/Phi-4-mini-instruct','revision':'cfbefacb99257ffa30c83adab238a50856ac3083','license_file':'LICENSE','license_sha256':'fa8235e5b48faca34e3ca98cf4f694ef08bd216d28b58071a1f85b1d50cb814d','families':['research_methodology','experimental_design','literature_synthesis','academic_articulation','research_ethics_reliability','scientific_verification']}}
PROMPT_VERSION='teacher-v1'
def teacher_messages(r):
    # No gold/reference response or benchmark is ever passed to the model.
    return [{'role':'system','content':'You are a scientific research tutor. Use only the supplied task and evidence. Label simulations, state assumptions, avoid invented citations or observations. Output one JSON object with explanation and final_value (a number for numeric tasks, null otherwise). Do not claim tool execution; describe a proposed procedure.'},{'role':'user','content':r['prompt']+'\nSupplied evidence: '+canon(r['evidence'])}]
def parse_proposal(text,r):
    try:
        obj=json.loads(text.strip())
        if not isinstance(obj,dict) or not isinstance(obj.get('explanation'),str) or not obj['explanation'].strip(): raise ValueError('invalid schema')
        value=obj.get('final_value')
        good=(isinstance(value,(int,float)) and not isinstance(value,bool) and math.isfinite(value) and close(value,r['independent_reference'])) if r['numeric_answer'] is not None else None
        return {'protocol_valid':True,'numeric_match':good,'explanation':obj['explanation'],'final_value':value if isinstance(value,(int,float)) and math.isfinite(value) else None}
    except (ValueError,TypeError,OverflowError): return {'protocol_valid':False,'numeric_match':None,'explanation':None,'final_value':None}
def route(spec):
    groups=defaultdict(list)
    for r in dev:
        if r['family'] in spec['families']: groups[r['template_id']].append(r)
    for g in groups.values(): random.Random(SEED).shuffle(g)
    result=[]
    while len(result)<TEACHER_LIMIT and any(groups.values()):
        for key in sorted(groups):
            if groups[key] and len(result)<TEACHER_LIMIT: result.append(groups[key].pop())
    return result
teacher_proposals=[]; teacher_errors=[]
# CPU tests of parser and routing (not actual teacher inference)
assert parse_proposal('{"explanation":"test","final_value":2}',{'numeric_answer':2,'independent_reference':2})['numeric_match'] is True
assert not parse_proposal('{"explanation":"test","final_value":true}',{'numeric_answer':2,'independent_reference':2})['numeric_match']
assert not parse_proposal('not JSON',dev[0])['protocol_valid']
assert all(r['split']=='development_candidate' for s in TEACHERS.values() for r in route(s))
assert RUN_TEACHERS and all(LICENSE_APPROVALS.values()), 'GPU teacher run and explicit approvals required.'
if RUN_TEACHERS:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.53,<5','accelerate','bitsandbytes','huggingface_hub','safetensors'])
    import torch,transformers,bitsandbytes
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
    from huggingface_hub import HfApi,hf_hub_download
    assert torch.cuda.is_available(),'GPU runtime required'
    db=sqlite3.connect(ROOT/'teacher_checkpoints.sqlite3')
    db.execute('CREATE TABLE IF NOT EXISTS results (key TEXT PRIMARY KEY, payload TEXT NOT NULL)')
    for name,spec in TEACHERS.items():
        if not LICENSE_APPROVALS.get(name): teacher_errors.append({'teacher':name,'error':'License/output-use approval not granted'}); continue
        tok=model=None
        try:
            info=HfApi().model_info(spec['repo'],revision=spec['revision'])
            assert info.sha==spec['revision'] and any(x.rfilename.endswith('.safetensors') for x in info.siblings)
            lic=Path(hf_hub_download(spec['repo'],spec['license_file'],revision=spec['revision']))
            assert sha(lic.read_bytes())==spec['license_sha256'],'License fingerprint mismatch'
            config={'corpus':CORPUS_HASH,'teacher':name,'revision':spec['revision'],'prompt_version':PROMPT_VERSION,'max_tokens':TEACHER_MAX_NEW_TOKENS,'quantization':'NF4-fp16','torch':torch.__version__,'transformers':transformers.__version__,'bnb':bitsandbytes.__version__,'gpu':torch.cuda.get_device_name(0)}
            pending=[]
            for r in route(spec):
                key=sha([config,r['id'],teacher_messages(r)])
                row=db.execute('SELECT payload FROM results WHERE key=?',(key,)).fetchone()
                if row: teacher_proposals.append(json.loads(row[0]))
                else: pending.append((r,key))
            if not pending: continue
            tok=AutoTokenizer.from_pretrained(spec['repo'],revision=spec['revision'],trust_remote_code=False)
            model=AutoModelForCausalLM.from_pretrained(spec['repo'],revision=spec['revision'],trust_remote_code=False,use_safetensors=True,device_map={'':0},torch_dtype=torch.float16,quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16))
            model.eval()
            for r,key in pending:
                try:
                    ids=tok.apply_chat_template(teacher_messages(r),tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
                    with torch.inference_mode(): out=model.generate(ids,attention_mask=torch.ones_like(ids),do_sample=False,max_new_tokens=TEACHER_MAX_NEW_TOKENS,pad_token_id=tok.eos_token_id)
                    text=tok.decode(out[0,ids.shape[-1]:],skip_special_tokens=True)
                    proposal={'record_id':r['id'],'teacher':name,'config':config,'raw_response':text,'assessment':parse_proposal(text,r),'training_approved':False,'human_decision':'pending'}
                    db.execute('INSERT INTO results VALUES (?,?)',(key,canon(proposal))); db.commit(); teacher_proposals.append(proposal)
                except Exception as e: teacher_errors.append({'teacher':name,'record_id':r['id'],'error':str(e)[:500]})
        except Exception as e: teacher_errors.append({'teacher':name,'error':str(e)[:500]})
        finally:
            del model,tok; gc.collect(); torch.cuda.empty_cache()
    db.close()
if teacher_proposals:
    screens=d07_screen([p['raw_response'] for p in teacher_proposals])
    for p,s in zip(teacher_proposals,screens):
        p['d07_response_screen']=s; p['disposition']='quarantine' if s['flagged'] or not p['assessment']['protocol_valid'] or p['assessment']['numeric_match'] is False else 'teacher_review_pending'
print('Teacher proposals:',len(teacher_proposals),'errors:',len(teacher_errors),'GPU requested:',RUN_TEACHERS)

## 9. Export candidate bundle and review gates
All records are private candidates; no SFT-ready files or auto-approval. Export source hashes, split manifest, exact dispositions, numerical and trace checks, review queue, duplicate/diversity audit, and unapproved teacher proposals separately. Do not include D07 questions, keys or model benchmark responses. Teacher checkpoint database remains in the private workspace and is not included. No D06 preference pairs are labeled automatically: reviewer must independently establish preferred/rejected quality and verify both answers. External literature evidence and source permissions remain future work.

In [ ]:
assert RUN_TEACHERS and all(LICENSE_APPROVALS.values()), 'GPU teacher run and explicit approvals required.'
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); OUT=ROOT/run_id; OUT.mkdir()
for d in ['candidate_datasets','quarantine','reports','teacher_review','tool_registry']: (OUT/d).mkdir()
for dataset in sorted(set(DATASETS.values())):
    for split in ['development_candidate','validation_candidate']:
        write_lines(OUT/'candidate_datasets'/f'{dataset}_{split}.jsonl',[r for r in records if r['dataset']==dataset and r['split']==split])
write_lines(OUT/'quarantine'/'records.jsonl',quarantine)
write_lines(OUT/'teacher_review'/'unapproved_proposals.jsonl',[p for p in teacher_proposals if p.get('disposition')!='quarantine'])
write_lines(OUT/'quarantine'/'teacher_proposals.jsonl',[p for p in teacher_proposals if p.get('disposition')=='quarantine'])
write_json(OUT/'teacher_review'/'errors.json',teacher_errors)
write_json(OUT/'tool_registry'/'D10_registry.json',{'tools':TOOL_NAMES,'scope':'bounded generator-owned synthetic calls only','schema':'tool_trace list with ordered step, arguments, depends_on, status, observation or error, replay_pass'})
write_lines(OUT/'reports'/'record_ledger.jsonl',[{k:r[k] for k in ['id','dataset','family','template_id','record_sha256','verification','split','disposition','d07_screen','prior_pilot_char_similarity','training_approved']} for r in records])
pd.DataFrame(diversity).to_csv(OUT/'reports'/'diversity.csv',index=False)
pd.DataFrame([{'record_id':r['id'],'family':r['family'],'split':r['split'],'prompt':r['prompt'],'reference_response':r['reference_response'],'rubric':canon(r['rubric']),'reviewer':'','decision':'pending','notes':''} for r in records]).to_csv(OUT/'reports'/'human_review.csv',index=False)
# Prioritize a bounded, template-balanced review set, not an approved training subset.
review_groups=defaultdict(list)
for r in dev+val: review_groups[r['template_id']].append(r)
representative=[]
for key,g in sorted(review_groups.items()):
    # Prefer distinct masked response shapes, then use deterministic seeded sampling.
    shuffled=list(g); random.Random(SEED+sum(map(ord,key))).shuffle(shuffled)
    picked=[]; shapes=set()
    for r in shuffled:
        shape=mask(r['reference_response'])
        if shape not in shapes and len(picked)<5: picked.append(r); shapes.add(shape)
    for r in shuffled:
        if len(picked)>=5: break
        if r['id'] not in {x['id'] for x in picked}: picked.append(r)
    representative.extend(picked)
pd.DataFrame([{'record_id':r['id'],'template_id':r['template_id'],'split':r['split'],'prompt':r['prompt'],'reference_response':r['reference_response'],'rubric':canon(r['rubric']),'decision':'pending','reviewer':'','notes':''} for r in representative]).to_csv(OUT/'reports'/'representative_review_first.csv',index=False)
# Pair overlapping teacher responses without declaring a winner or generating DPO labels.
proposals_by_id=defaultdict(list)
for p in teacher_proposals: proposals_by_id[p['record_id']].append(p)
write_lines(OUT/'teacher_review'/'comparison_review.jsonl',[{'record_id':rid,'proposals':ps,'preferred_teacher':None,'decision':'pending','reason':''} for rid,ps in proposals_by_id.items() if len(ps)>1 and all(p.get('disposition')!='quarantine' for p in ps)])
write_json(OUT/'reports'/'split.json',{'validation_templates':sorted(val_templates),'group_overlap':0,'counts':dict(Counter(r['split'] for r in records)),'status':'provisional semantic review required'})
summary={'version':VERSION,'seed':SEED,'corpus_sha256':CORPUS_HASH,'generated':len(records),'family_counts':dict(Counter(r['family'] for r in records)),'dataset_counts':dict(Counter(r['dataset'] for r in records)),'template_count':len({r['template_id'] for r in records}),'representative_review_count':len(representative),'unique_exact_responses':len({norm(r['reference_response']) for r in records}),'unique_number_masked_responses':len({mask(r['reference_response']) for r in records}),'records_with_near_masked_prompt_095':sum(r['diversity']['nearest_masked_prompt']>=.95 for r in records),'numeric_checks':sum(r['verification']['passed'] is not None for r in records),'numeric_pass':sum(r['verification']['passed'] is True for r in records),'narrative_pending':sum(r['verification']['passed'] is None for r in records),'multi_tool_workflows':sum(bool(r['tool_trace']) for r in records),'executed_steps':sum(s['status']=='executed' for r in records for s in (r['tool_trace'] or [])),'rejected_steps':sum(s['status']=='rejected' for r in records for s in (r['tool_trace'] or [])),'replay_pass_steps':sum(s['replay_pass'] for r in records for s in (r['tool_trace'] or [])),'d07':{'benchmark_sha256':EXPECTED_BENCHMARK,'questions_sha256':EXPECTED_QUESTIONS,'questions_screened':1000,'flagged':sum(x['flagged'] for x in leak),'max_char':max(x['char'] for x in leak),'max_word':max(x['word'] for x in leak),'gold_keys_read':False,'scope':'lexical only'},'split_counts':dict(Counter(r['split'] for r in records)),'semantic_status':semantic_status,'teacher_requested':RUN_TEACHERS,'teacher_proposals':len(teacher_proposals),'teacher_errors':len(teacher_errors),'rejected_attempts':dict(rejected_attempts),'training_approved':False,'publishing_approved':False,'runtime':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'sympy':sp.__version__,'sklearn':sklearn.__version__}}
expected_by_teacher={name:len(route(spec)) for name,spec in TEACHERS.items()}
actual_by_teacher=dict(Counter(p['teacher'] for p in teacher_proposals))
complete=all(actual_by_teacher.get(name,0)==count and count==TEACHER_LIMIT for name,count in expected_by_teacher.items()) and not teacher_errors
summary.update({'teacher_expected_by_model':expected_by_teacher,'teacher_actual_by_model':actual_by_teacher,'teacher_run_complete':complete,'teacher_run_status':'complete' if complete else 'partial_or_failed'})
write_json(OUT/'reports'/'summary.json',summary)
write_json(OUT/'reports'/'release_gates.json',{'generation_complete':True,'human_review_complete':False,'semantic_contamination_excluded':False,'source_licensing_approved':False,'teacher_outputs_approved':False,'native_qwen_tool_format_validated':False,'independent_sealed_confirmation_benchmark_ready':False,'training_approved':False,'publication_approved':False})
(OUT/'README.md').write_text('Trimorious M1.4.1 private candidate corpus. Synthetic cases only. Not training-ready. D07 never included. Teacher proposals are separate and unapproved. See reports/summary.json and release_gates.json.\n')
manifest={str(f.relative_to(OUT)):sha(f.read_bytes()) for f in sorted(OUT.rglob('*')) if f.is_file()}; write_json(OUT/'archive_manifest.json',manifest)
ZIP=ROOT/f'trimorious_m1_4_1_PRIVATE_GPU_{"complete" if complete else "partial"}_{run_id}.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file(): z.write(f,f.relative_to(OUT))
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None and all(sha(z.read(n))==h for n,h in manifest.items())
    assert not any('D07_questions' in n or 'D07_gold' in n for n in z.namelist())
print(json.dumps(summary,indent=2)); print('Manifest:',len(manifest),'files verified'); print('PRIVATE ZIP:',ZIP)
print('GPU pilot complete:',complete, '| per-model proposals:',actual_by_teacher, '| expected:',expected_by_teacher)
if not complete: print('WARNING: partial or failed teacher run. Inspect teacher_review/errors.json; upload this NEW GPU archive for diagnosis.')

## 10. Exit gates and next run
- Review all numerical derivations, units, trace semantics and case-specific narrative rubrics.
- Resolve D07 flags and cross-version overlaps; no automatic replacement or threshold lowering.
- Review nearest-neighbor and masked-answer reports; template counts are not counts of independent intellectual skills.
- Run teachers only after license/output-use approval. Independently verify proposals; retain original reference and record decisions.
- Add genuinely licensed research evidence, multilingual cases, native Qwen3 tool format, and diverse human-authored tasks before expansion.
- Keep D07 private; commission a separate sealed test using independently authored tasks and a separate custodian. Do not use its answers to develop training data.
- M1.5 requires human approvals, source/redistribution checks, dataset cards, and an explicit publication decision. M2.0 requires training approval.

In [ ]:
DOWNLOAD_ZIP=True
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(ZIP))